# DATA MATCHING

## Mục tiêu
- Thu thập danh sách ảnh từ 4 thư mục dữ liệu.
- Đọc và chuẩn hóa Metadata.
- Ghép ảnh với Metadata dựa trên quy tắc đánh số của GVLiD.
- Kiểm tra các bản ghi được ghép và chưa được ghép.
- Xuất dataset hoàn chỉnh dưới dạng CSV.

**1. IMPORT THƯ VIỆN**

In [1]:
import json
import re
import pandas as pd
from pathlib import Path

print("Import thư viện thành công!")

Import thư viện thành công!


**2. Kết nối dữ liệu**

Đọc Metadata từ file JSON và chuyển đổi thành DataFrame
để thuận tiện cho việc xử lý và đối chiếu.

In [2]:
# Đường dẫn đến thư mục chứa GVLiD
DATASET_DIR = Path(r"D:\DataMining26\Smart-Vineyard\Dataset\GVLiD GrapeVine Leaf identification of the Disease\DATASET")

# Đường dẫn lưu kết quả
OUTPUT_DIR = Path.cwd().parent / "outputs" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Kiểm tra đường dẫn
assert DATASET_DIR.exists(), "Không tìm thấy thư mục Dataset!"

# Đọc Metadata
json_path = DATASET_DIR / "metadata.json"

with open(json_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

# Chuyển JSON thành DataFrame
df_metadata = pd.json_normalize(metadata)

# Loại bỏ 2 bản ghi không có ảnh
df_metadata_clean = df_metadata[
    ~df_metadata["Sr_No"].isin([20, 809])
].copy()

print("Số Metadata ban đầu:", len(df_metadata))
print("Số Metadata sau làm sạch:", len(df_metadata_clean))

display(df_metadata_clean.head())

Số Metadata ban đầu: 3478
Số Metadata sau làm sạch: 3476


,Sr_No,Plant_Health_Status,Disease_Type,Location.Vineyard,Location.GPS.Latitude,Location.GPS.Longitude,Environmental_Conditions.Time_of_Capture,Environmental_Conditions.Lighting,Environmental_Conditions.Weather,Device_Specifications.Device_Name,Device_Specifications.Sensor,Device_Specifications.OS
0,1,Diseased,Black Rot,Vineyard_22,18.785238,73.994949,Afternoon,Overcast,Humid,Redmi Note 10 5G,Sony IMX 582 1/2rr,Android
1,2,Diseased,Black Rot,Vineyard_28,18.731029,73.977206,Afternoon,Natural sunlight,Humid,Redmi Note 10 5G,Sony IMX 582 1/2rr,Android
2,3,Diseased,Black Rot,Vineyard_14,18.802297,74.027516,Afternoon,Partly cloudy,Humid,Redmi Note 10 5G,Sony IMX 582 1/2rr,Android
3,4,Diseased,Black Rot,Vineyard_20,18.775820,74.000550,Morning,Overcast,Humid,Redmi Note 10 5G,Sony IMX 582 1/2rr,Android
4,5,Diseased,Black Rot,Vineyard_8,18.790580,74.002429,Evening,Partly cloudy,Clear,Redmi Note 10 5G,Sony IMX 582 1/2rr,Android


**3. Thu thập danh sách hình ảnh**

Duyệt qua 4 thư mục lớp:
- Black rot
- esca
- healthy
- leaf blight

Trích xuất số thứ tự từ tên file để sử dụng làm khóa ghép dữ liệu.

In [3]:
# Ánh xạ tên thư mục với nhãn bệnh trong Metadata
class_mapping = {
    "Black rot": "Black Rot",
    "esca": "Esca",
    "healthy": "Healthy",
    "leaf blight": "Leaf Blight"
}

image_rows = []

for folder_name, disease_name in class_mapping.items():

    folder_path = DATASET_DIR / folder_name

    # Duyệt các file ảnh trong thư mục
    for image_path in folder_path.glob("*"):

        if not image_path.is_file():
            continue

        # Chỉ nhận file có dạng tên lớp + số thứ tự
        match = re.fullmatch(
            rf"{re.escape(folder_name)}(\d+)",
            image_path.stem,
            flags=re.IGNORECASE
        )

        if match is None:
            continue

        image_no = int(match.group(1))

        image_rows.append({
            "Image_Name": image_path.name,
            "Image_Path": str(image_path),
            "Disease_Type": disease_name,
            "Image_No": image_no
        })

# Tạo DataFrame ảnh
df_images = pd.DataFrame(image_rows)

# Sắp xếp dữ liệu
df_images = df_images.sort_values(
    ["Disease_Type", "Image_No"]
).reset_index(drop=True)

print("Tổng số ảnh:", len(df_images))
display(df_images.head())

Tổng số ảnh: 3476


,Image_Name,Image_Path,Disease_Type,Image_No
0,black rot001.jpg,D:\DataMining26\Smart-Vineyard\Dataset\GVLiD G...,Black Rot,1
1,black rot002.jpg,D:\DataMining26\Smart-Vineyard\Dataset\GVLiD G...,Black Rot,2
2,black rot003.jpg,D:\DataMining26\Smart-Vineyard\Dataset\GVLiD G...,Black Rot,3
3,black rot004.jpg,D:\DataMining26\Smart-Vineyard\Dataset\GVLiD G...,Black Rot,4
4,black rot005.jpg,D:\DataMining26\Smart-Vineyard\Dataset\GVLiD G...,Black Rot,5


## 4. Ghép hình ảnh với Metadata

Quy tắc ghép:
- Black Rot: sử dụng Sr_No làm khóa đối chiếu.
- Esca, Healthy, Leaf Blight: sử dụng thứ tự bản ghi trong từng lớp.

Sau đó ghép hai DataFrame dựa trên Disease_Type và Match_No.

In [4]:
# Tạo bản Metadata dùng để ghép
df_meta_match = df_metadata_clean.copy()

# Đánh số thứ tự trong từng lớp bệnh
df_meta_match["Match_No"] = (
    df_meta_match.groupby("Disease_Type").cumcount() + 1
)

# Riêng Black Rot: số ảnh tương ứng trực tiếp với Sr_No
df_meta_match.loc[
    df_meta_match["Disease_Type"] == "Black Rot",
    "Match_No"
] = df_meta_match.loc[
    df_meta_match["Disease_Type"] == "Black Rot",
    "Sr_No"
]

df_meta_match["Match_No"] = df_meta_match["Match_No"].astype(int)

# Tạo khóa tương ứng cho ảnh
df_images["Match_No"] = df_images["Image_No"]

# Ghép ảnh với Metadata
df_matched = pd.merge(
    df_images,
    df_meta_match,
    on=["Disease_Type", "Match_No"],
    how="outer",
    indicator=True
)

print("Kết quả ghép dữ liệu:")
print(df_matched["_merge"].value_counts())

display(df_matched.head())

Kết quả ghép dữ liệu:
_merge
both          3476
left_only        0
right_only       0
Name: count, dtype: int64


,Image_Name,Image_Path,Disease_Type,Image_No,Match_No,Sr_No,Plant_Health_Status,Location.Vineyard,Location.GPS.Latitude,Location.GPS.Longitude,Environmental_Conditions.Time_of_Capture,Environmental_Conditions.Lighting,Environmental_Conditions.Weather,Device_Specifications.Device_Name,Device_Specifications.Sensor,Device_Specifications.OS,_merge
0,black rot001.jpg,D:\DataMining26\Smart-Vineyard\Dataset\GVLiD G...,Black Rot,1,1,1,Diseased,Vineyard_22,18.785238,73.994949,Afternoon,Overcast,Humid,Redmi Note 10 5G,Sony IMX 582 1/2rr,Android,both
1,black rot002.jpg,D:\DataMining26\Smart-Vineyard\Dataset\GVLiD G...,Black Rot,2,2,2,Diseased,Vineyard_28,18.731029,73.977206,Afternoon,Natural sunlight,Humid,Redmi Note 10 5G,Sony IMX 582 1/2rr,Android,both
2,black rot003.jpg,D:\DataMining26\Smart-Vineyard\Dataset\GVLiD G...,Black Rot,3,3,3,Diseased,Vineyard_14,18.802297,74.027516,Afternoon,Partly cloudy,Humid,Redmi Note 10 5G,Sony IMX 582 1/2rr,Android,both
3,black rot004.jpg,D:\DataMining26\Smart-Vineyard\Dataset\GVLiD G...,Black Rot,4,4,4,Diseased,Vineyard_20,18.775820,74.000550,Morning,Overcast,Humid,Redmi Note 10 5G,Sony IMX 582 1/2rr,Android,both
4,black rot005.jpg,D:\DataMining26\Smart-Vineyard\Dataset\GVLiD G...,Black Rot,5,5,5,Diseased,Vineyard_8,18.790580,74.002429,Evening,Partly cloudy,Clear,Redmi Note 10 5G,Sony IMX 582 1/2rr,Android,both


## 5. Kiểm tra kết quả Data Matching

Kiểm tra:
- Số ảnh và số Metadata được ghép thành công.
- Ảnh không có Metadata.
- Metadata không có ảnh.
- Khóa ghép có bị trùng hay không.

In [5]:
# Thống kê trạng thái ghép
print("Số ảnh và Metadata ghép thành công:",
      (df_matched["_merge"] == "both").sum())

print("Số ảnh không có Metadata:",
      (df_matched["_merge"] == "left_only").sum())

print("Số Metadata không có ảnh:",
      (df_matched["_merge"] == "right_only").sum())

# Kiểm tra khóa ghép trùng
print("Số khóa ảnh trùng:",
      df_images.duplicated(
          ["Disease_Type", "Match_No"]
      ).sum())

print("Số khóa Metadata trùng:",
      df_meta_match.duplicated(
          ["Disease_Type", "Match_No"]
      ).sum())

# Thống kê số lượng theo lớp
display(
    df_matched[df_matched["_merge"] == "both"]
    .groupby("Disease_Type")
    .size()
    .rename("So_luong")
    .to_frame()
)

# Lọc các bản ghi ghép thành công
df_final = df_matched[
    df_matched["_merge"] == "both"
].copy()

# Chỉ xuất khi toàn bộ dữ liệu đã ghép thành công
if (
    len(df_final) == 3476
    and (df_matched["_merge"] != "both").sum() == 0
    and not df_final.duplicated(
        ["Disease_Type", "Match_No"]
    ).any()
):
    output_path = OUTPUT_DIR / "image_metadata.csv"

    df_final.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("Đã xuất dữ liệu thành công!")
    print("Đường dẫn:", output_path)

else:
    print("Dữ liệu chưa đạt điều kiện. Cần kiểm tra lại!")

Số ảnh và Metadata ghép thành công: 3476
Số ảnh không có Metadata: 0
Số Metadata không có ảnh: 0
Số khóa ảnh trùng: 0
Số khóa Metadata trùng: 0


,So_luong
Disease_Type,
Black Rot,807
Esca,888
Healthy,1109
Leaf Blight,672


Đã xuất dữ liệu thành công!
Đường dẫn: c:\Users\Khang Nguyen\AppData\Local\Programs\outputs\processed\image_metadata.csv
